# V5：Stage2 Transformer Baseline

本 Notebook 基于老师给的 `V1+V2` 框架，新增一个真正的 Transformer 序列模型实验。目标是检验：在 V2 严格因果、walk-forward、统一 evaluator 的口径下，Transformer / Multi-Head Attention 是否能利用历史特征序列，提高个股未来收益预测效果。

默认任务：预测下一交易日收益 `label_1d_raw`。如需对齐 V1 的未来 5 日收益任务，可以在配置 Cell 中把 `LABEL_HORIZON` 改成 `"5d"`。

运行前请确认：

1. `PROJECT_ROOT` 指向你本地的 `V1+V2/V2` 文件夹。
2. V2 的数据缓存已经能被原项目读取。
3. 先运行原项目必要的数据准备或 baseline notebook，确保 `daily_panel_with_labels.parquet` 等缓存存在。


In [1]:
from pathlib import Path
import os
import sys
import json
import math
import time
import copy
import logging
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Pandas:", pd.__version__)


Python: 3.11.5 (main, Sep 11 2023, 08:31:25) [Clang 14.0.6 ]
Torch: 2.7.1
Pandas: 2.0.3


## 1. 项目路径与运行配置

把下面的 `PROJECT_ROOT` 改成本地 `V1+V2/V2` 的真实路径。默认路径：

```text
/Users/runtianzhou/Turing_AI_v5/V1+V2/V2
```


In [3]:
# =============================
# 需要重点检查这个路径
# =============================
PROJECT_ROOT = Path("/Users/runtianzhou/Turing_AI_v5/V1+V2/V2")

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
REPORT_DIR = PROJECT_ROOT / "reports"
FIGURE_DIR = ARTIFACT_DIR / "figures"

for d in [ARTIFACT_DIR, REPORT_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_ROOT:", SRC_ROOT)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("REPORT_DIR:", REPORT_DIR)
print("Exists PROJECT_ROOT:", PROJECT_ROOT.exists())
print("Exists SRC_ROOT:", SRC_ROOT.exists())


PROJECT_ROOT: /Users/runtianzhou/Turing_AI_v5/V1+V2/V2
SRC_ROOT: /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/src
ARTIFACT_DIR: /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/artifacts
REPORT_DIR: /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/reports
Exists PROJECT_ROOT: True
Exists SRC_ROOT: True


In [4]:
# =============================
# 导入 V2 原项目模块
# =============================
from evaluation.evaluator import evaluate_predictions, plot_evaluation
from models import ridge_baseline as v1
from models import ridge_baseline_v2 as v2

# =============================
# Transformer 实验配置
# =============================
LABEL_HORIZON = "1d"   # 可改成 "5d"，用于预测未来 5 日收益
assert LABEL_HORIZON in {"1d", "5d"}

LABEL_COL = f"label_{LABEL_HORIZON}_raw"
LABEL_TIME_COL = f"label_time_{LABEL_HORIZON}"

FEATURE_COLUMNS = list(v2.FEATURE_COLUMNS)

SEQUENCE_LENGTH = 20       # 每只股票使用过去 20 个交易观察点作为输入序列
D_MODEL = 64               # Transformer hidden dimension
N_HEADS = 4                # Multi-Head Attention heads
N_LAYERS = 2               # Transformer Encoder 层数
DROPOUT = 0.10

# 为了先跑通，默认每个 rolling window 最多抽样 120000 条训练序列
# 如果电脑性能足够，可以改成 0，表示使用全部训练样本
MAX_TRAIN_SAMPLES_PER_WINDOW = 120000
MAX_EPOCHS = 12
PATIENCE = 3
MIN_DELTA = 1e-7
BATCH_SIZE = 2048
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VALIDATION_DATE_FRACTION = 0.10
RANDOM_STATE = 20260812

PREDICTIONS_PATH = ARTIFACT_DIR / f"transformer_{LABEL_HORIZON}_predictions.parquet"
WINDOWS_PATH = ARTIFACT_DIR / f"transformer_{LABEL_HORIZON}_windows.parquet"
SUMMARY_PATH = REPORT_DIR / f"transformer_{LABEL_HORIZON}_summary.md"
LOSS_FIGURE_PATH = FIGURE_DIR / f"transformer_{LABEL_HORIZON}_loss_curves.png"

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

# Mac MPS 偶尔对某些算子不稳定；如果遇到 device 报错，可以手动改成：
# DEVICE = torch.device("cpu")

print("LABEL_HORIZON:", LABEL_HORIZON)
print("LABEL_COL:", LABEL_COL)
print("LABEL_TIME_COL:", LABEL_TIME_COL)
print("FEATURE_COLUMNS:", FEATURE_COLUMNS)
print("N_FEATURES:", len(FEATURE_COLUMNS))
print("DEVICE:", DEVICE)
print("PREDICTIONS_PATH:", PREDICTIONS_PATH)
print("SUMMARY_PATH:", SUMMARY_PATH)


LABEL_HORIZON: 1d
LABEL_COL: label_1d_raw
LABEL_TIME_COL: label_time_1d
FEATURE_COLUMNS: ['return_5d', 'return_10d', 'return_20d', 'turnover_20d_mean', 'log_neg_market_value']
N_FEATURES: 5
DEVICE: mps
PREDICTIONS_PATH: /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/artifacts/transformer_1d_predictions.parquet
SUMMARY_PATH: /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/reports/transformer_1d_summary.md


## 2. 读取 V2 panel，并构造严格 T-1 特征

这里复用 V2 的 `ridge_baseline_v2.build_features`，这样 Transformer 的输入特征口径与 V2 修正版 Ridge baseline 保持一致。


In [6]:
# 自动生成 V2 data_cache

# =========================
# 生成 V2 data_cache
# 必须放在 v1.load_panel() 前面运行
# =========================

from pathlib import Path
import os
import sys
import shutil
import subprocess

# 你的 V2 项目根目录
PROJECT_ROOT = Path("/Users/runtianzhou/Turing_AI_v5/V1+V2/V2")

# V2 的父目录，也就是 V1+V2
PACKAGE_ROOT = PROJECT_ROOT.parent

# V2 代码默认寻找这里：
# /Users/runtianzhou/Turing_AI_v5/V1+V2/data_source/daily_K
EXPECTED_DAILY_K_DIR = PACKAGE_ROOT / "data_source" / "daily_K"

# 你的原始日线 CSV 可能在这些位置
SOURCE_DAILY_CANDIDATES = [
    Path("/Users/runtianzhou/Turing_AI/data_extracted/daily_temp3"),
    Path("/Users/runtianzhou/Turing_AI_v2/data_extracted/daily_temp3"),
    Path("/Users/runtianzhou/Turing_AI_v4/data_extracted/daily_temp3"),
    Path("/Users/runtianzhou/Turing_AI_v5/data_extracted/daily_temp3"),
    PACKAGE_ROOT / "data_source" / "daily_K",
]

DATA_CACHE_DIR = PROJECT_ROOT / "data_cache"
DAILY_PANEL_PATH = DATA_CACHE_DIR / "daily_panel.parquet"
LABELED_PANEL_PATH = DATA_CACHE_DIR / "daily_panel_with_labels.parquet"

DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_DAILY_K_DIR.parent.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PACKAGE_ROOT:", PACKAGE_ROOT)
print("EXPECTED_DAILY_K_DIR:", EXPECTED_DAILY_K_DIR)
print("DAILY_PANEL_PATH:", DAILY_PANEL_PATH)
print("LABELED_PANEL_PATH:", LABELED_PANEL_PATH)

# 1. 确认原始 daily_K 数据目录
if not EXPECTED_DAILY_K_DIR.exists():
    source_daily_dir = None

    for p in SOURCE_DAILY_CANDIDATES:
        if p.exists() and len(list(p.glob("*.csv"))) > 0:
            source_daily_dir = p
            break

    if source_daily_dir is None:
        raise FileNotFoundError(
            "没有找到原始日线CSV目录。请确认 daily_temp3 在哪里，"
            "然后把 SOURCE_DAILY_CANDIDATES 里的路径改成你的实际路径。"
        )

    print("找到原始日线数据:", source_daily_dir)

    # 优先创建软链接，避免复制几GB数据
    try:
        os.symlink(source_daily_dir, EXPECTED_DAILY_K_DIR, target_is_directory=True)
        print("已创建软链接:")
        print(EXPECTED_DAILY_K_DIR, "->", source_daily_dir)
    except FileExistsError:
        print("daily_K 目录已存在:", EXPECTED_DAILY_K_DIR)
    except Exception as e:
        print("软链接失败，改为复制。原因:", e)
        shutil.copytree(source_daily_dir, EXPECTED_DAILY_K_DIR, dirs_exist_ok=True)
        print("已复制日线数据到:", EXPECTED_DAILY_K_DIR)

else:
    print("V2 daily_K 目录已存在:", EXPECTED_DAILY_K_DIR)

print("daily_K CSV 数量:", len(list(EXPECTED_DAILY_K_DIR.glob("*.csv"))))

# 2. 生成 daily_panel.parquet
if not DAILY_PANEL_PATH.exists():
    print("\n开始生成 daily_panel.parquet ...")

    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "data" / "loaders.py"),
        "--start-date", "2020-01-02",
        "--end-date", "2026-04-09",
        "--output", str(DAILY_PANEL_PATH),
    ]

    print("运行命令:")
    print(" ".join(cmd))

    subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

else:
    print("\ndaily_panel.parquet 已存在，跳过生成。")

# 3. 生成 daily_panel_with_labels.parquet
if not LABELED_PANEL_PATH.exists():
    print("\n开始生成 daily_panel_with_labels.parquet ...")

    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "labels" / "label_builder.py"),
        "--input", str(DAILY_PANEL_PATH),
        "--output", str(LABELED_PANEL_PATH),
    ]

    print("运行命令:")
    print(" ".join(cmd))

    subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

else:
    print("\ndaily_panel_with_labels.parquet 已存在，跳过生成。")

print("\n完成。现在可以运行 v1.load_panel()。")
print("存在 daily_panel_with_labels:", LABELED_PANEL_PATH.exists())

PROJECT_ROOT: /Users/runtianzhou/Turing_AI_v5/V1+V2/V2
PACKAGE_ROOT: /Users/runtianzhou/Turing_AI_v5/V1+V2
EXPECTED_DAILY_K_DIR: /Users/runtianzhou/Turing_AI_v5/V1+V2/data_source/daily_K
DAILY_PANEL_PATH: /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/data_cache/daily_panel.parquet
LABELED_PANEL_PATH: /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/data_cache/daily_panel_with_labels.parquet
找到原始日线数据: /Users/runtianzhou/Turing_AI/data_extracted/daily_temp3
已创建软链接:
/Users/runtianzhou/Turing_AI_v5/V1+V2/data_source/daily_K -> /Users/runtianzhou/Turing_AI/data_extracted/daily_temp3
daily_K CSV 数量: 1476

开始生成 daily_panel.parquet ...
运行命令:
/opt/homebrew/anaconda3/bin/python3 /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/src/data/loaders.py --start-date 2020-01-02 --end-date 2026-04-09 --output /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/data_cache/daily_panel.parquet
[load]    1/1476 files: 20200102.csv
[load]  100/1476 files: 20200603.csv
[load]  200/1476 files: 20201102.csv
[load]  300/1476 files: 20210330.csv

In [7]:
# 读取 V2 panel
panel = v1.load_panel()

print("Panel shape:", panel.shape)
print("Panel columns sample:", list(panel.columns)[:30])
print("Date range:", panel["tradeDate"].min(), "to", panel["tradeDate"].max())
print("Stocks:", panel["secID"].nunique())

if LABEL_COL not in panel.columns or LABEL_TIME_COL not in panel.columns:
    raise ValueError(
        f"Panel does not contain {LABEL_COL}/{LABEL_TIME_COL}. "
        "请先运行 V2 的标签构造流程，确保 daily_panel_with_labels.parquet 已包含对应标签。"
    )

# 构造严格滞后一日的信息集特征
featured = v2.build_features(panel)

print("Featured shape:", featured.shape)
print("Feature columns:", FEATURE_COLUMNS)
print("Missing feature rows:", featured[FEATURE_COLUMNS].isna().any(axis=1).sum())
print("Label non-null rows:", featured[LABEL_COL].notna().sum())
featured[["secID", "tradeDate", LABEL_COL, LABEL_TIME_COL, "predict_time", "feature_input_end_date"] + FEATURE_COLUMNS].head()


Panel shape: (6609016, 8)
Panel columns sample: ['secID', 'tradeDate', 'adj_close', 'turnoverRate', 'negMarketValue', 'label_1d_raw', 'predict_time', 'label_time_1d']
Date range: 2020-01-02 00:00:00 to 2026-02-02 00:00:00
Stocks: 5378
Featured shape: (6609016, 14)
Feature columns: ['return_5d', 'return_10d', 'return_20d', 'turnover_20d_mean', 'log_neg_market_value']
Missing feature rows: 112673
Label non-null rows: 6603549


,secID,tradeDate,label_1d_raw,label_time_1d,predict_time,feature_input_end_date,return_5d,return_10d,return_20d,turnover_20d_mean,log_neg_market_value
0,000001.XSHE,2020-01-02,0.018376,2020-01-03 15:00:00,2020-01-02 15:00:00,NaT,NaN,NaN,NaN,NaN,NaN
3551,000001.XSHE,2020-01-03,-0.006403,2020-01-06 15:00:00,2020-01-03 15:00:00,2020-01-02,NaN,NaN,NaN,NaN,26.514372
7115,000001.XSHE,2020-01-06,0.004687,2020-01-07 15:00:00,2020-01-06 15:00:00,2020-01-03,NaN,NaN,NaN,NaN,26.532581
10660,000001.XSHE,2020-01-07,-0.028571,2020-01-08 15:00:00,2020-01-07 15:00:00,2020-01-06,NaN,NaN,NaN,NaN,26.526157
14221,000001.XSHE,2020-01-08,0.007803,2020-01-09 15:00:00,2020-01-08 15:00:00,2020-01-07,NaN,NaN,NaN,NaN,26.530834


## 3. Dataset：按个股构造历史序列

每个样本为某只股票在某一天的预测任务：

```text
过去 20 个交易观察点的特征序列 -> Transformer -> 预测未来收益
```

注意：V2 特征本身已经由 `build_features` 保证使用 T-1 及更早信息，因此序列最后一个 token 仍满足严格因果口径。


In [8]:
class SequenceDataset(Dataset):
    """Per-stock rolling sequence dataset."""

    def __init__(
        self,
        frame: pd.DataFrame,
        allowed_mask: pd.Series,
        scaler: StandardScaler,
        target_scale: float,
        label_lower: float | None = None,
        label_upper: float | None = None,
        require_label: bool = True,
        max_samples: int | None = None,
        random_state: int = RANDOM_STATE,
    ) -> None:
        if not frame.index.equals(allowed_mask.index):
            raise ValueError("allowed_mask must align with frame index")

        needed = [
            "secID",
            "tradeDate",
            "predict_time",
            LABEL_TIME_COL,
            LABEL_COL,
            "feature_input_end_date",
            *FEATURE_COLUMNS,
        ]
        missing = set(needed).difference(frame.columns)
        if missing:
            raise ValueError(f"Frame is missing columns: {sorted(missing)}")

        data = frame[needed].copy()
        data["_allowed"] = allowed_mask.astype(bool).to_numpy()
        data.sort_values(["secID", "tradeDate"], kind="mergesort", inplace=True)
        data.reset_index(drop=True, inplace=True)

        raw_x = data[FEATURE_COLUMNS].to_numpy(dtype="float64")
        row_feature_ok = np.isfinite(raw_x).all(axis=1)
        scaled_x = scaler.transform(raw_x).astype("float32")

        self.groups: Dict[str, Dict[str, object]] = {}
        self.samples: List[Tuple[str, int]] = []
        self.meta: List[Dict[str, object]] = []
        self.target_scale = float(target_scale)
        self.label_lower = label_lower
        self.label_upper = label_upper
        self.require_label = require_label

        for secid, g in data.groupby("secID", sort=False, observed=True):
            idx = g.index.to_numpy()
            x = scaled_x[idx]
            ok = row_feature_ok[idx]
            allowed = g["_allowed"].to_numpy(dtype=bool)
            y_raw = g[LABEL_COL].to_numpy(dtype="float64")
            trade_dates = pd.to_datetime(g["tradeDate"]).to_numpy()
            predict_times = pd.to_datetime(g["predict_time"]).to_numpy()
            label_times = pd.to_datetime(g[LABEL_TIME_COL]).to_numpy()
            feature_end_dates = pd.to_datetime(g["feature_input_end_date"]).to_numpy()

            self.groups[str(secid)] = {
                "x": x,
                "ok": ok,
                "allowed": allowed,
                "y_raw": y_raw,
                "trade_dates": trade_dates,
                "predict_times": predict_times,
                "label_times": label_times,
                "feature_end_dates": feature_end_dates,
            }

            for i in range(SEQUENCE_LENGTH - 1, len(g)):
                if not allowed[i]:
                    continue
                if not ok[i - SEQUENCE_LENGTH + 1 : i + 1].all():
                    continue
                if require_label and not np.isfinite(y_raw[i]):
                    continue
                if pd.isna(feature_end_dates[i]):
                    continue
                # 核心因果检查：最后一个输入特征的结束日必须早于预测日
                if pd.Timestamp(feature_end_dates[i]) >= pd.Timestamp(predict_times[i]).normalize():
                    continue
                self.samples.append((str(secid), i))

        if max_samples is not None and max_samples > 0 and len(self.samples) > max_samples:
            rng = np.random.default_rng(random_state)
            chosen = rng.choice(len(self.samples), size=max_samples, replace=False)
            self.samples = [self.samples[i] for i in np.sort(chosen)]

        for secid, i in self.samples:
            g = self.groups[secid]
            self.meta.append(
                {
                    "secID": secid,
                    "tradeDate": pd.Timestamp(g["trade_dates"][i]),
                    "predict_time": pd.Timestamp(g["predict_times"][i]),
                    "label_time": pd.Timestamp(g["label_times"][i]) if not pd.isna(g["label_times"][i]) else pd.NaT,
                    "y_true": float(g["y_raw"][i]) if np.isfinite(g["y_raw"][i]) else np.nan,
                    "feature_input_end_date": pd.Timestamp(g["feature_end_dates"][i]),
                }
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, sample_id: int) -> Dict[str, torch.Tensor]:
        secid, i = self.samples[sample_id]
        g = self.groups[secid]
        seq = g["x"][i - SEQUENCE_LENGTH + 1 : i + 1]
        y = float(g["y_raw"][i]) if np.isfinite(g["y_raw"][i]) else np.nan
        if self.label_lower is not None and self.label_upper is not None and np.isfinite(y):
            y = float(np.clip(y, self.label_lower, self.label_upper))
        y_scaled = y / self.target_scale if np.isfinite(y) else np.nan
        return {
            "x": torch.from_numpy(seq),
            "y": torch.tensor(y_scaled, dtype=torch.float32),
            "sample_id": torch.tensor(sample_id, dtype=torch.long),
        }


## 4. Transformer 模型

结构：

```text
历史特征序列 -> Linear Projection -> Positional Encoding -> Transformer Encoder -> 最后一个 token -> 回归头 -> 预测收益
```


In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 0:
            pe[:, 1::2] = torch.cos(position * div_term)
        else:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]


class TransformerRegressor(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.input_proj = nn.Linear(input_dim, D_MODEL)
        self.pos = PositionalEncoding(D_MODEL, max_len=max(512, SEQUENCE_LENGTH + 4))
        layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dim_feedforward=D_MODEL * 4,
            dropout=DROPOUT,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
        self.head = nn.Sequential(
            nn.LayerNorm(D_MODEL),
            nn.Linear(D_MODEL, D_MODEL),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(D_MODEL, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.input_proj(x)
        h = self.pos(h)
        h = self.encoder(h)
        last = h[:, -1, :]
        return self.head(last).squeeze(-1)

model_preview = TransformerRegressor(input_dim=len(FEATURE_COLUMNS))
model_preview


TransformerRegressor(
  (input_proj): Linear(in_features=5, out_features=64, bias=True)
  (pos): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=64, out_features=64, bias=True)
    (2): GELU(approximate='none')
    (

## 5. 训练、预测与评估函数

In [10]:
def mean_loss(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    total = 0.0
    n = 0
    criterion = nn.MSELoss(reduction="sum")
    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(DEVICE)
            y = batch["y"].to(DEVICE)
            pred = model(x)
            loss = criterion(pred, y)
            total += float(loss.item())
            n += len(y)
    return total / max(n, 1)


def fit_model(train_ds: SequenceDataset, val_ds: SequenceDataset, window_id: int):
    torch.manual_seed(RANDOM_STATE + window_id)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(RANDOM_STATE + window_id)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)

    model = TransformerRegressor(input_dim=len(FEATURE_COLUMNS)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.MSELoss()

    train_losses = []
    val_losses = []
    best_loss = math.inf
    best_epoch = 0
    best_state = None
    stale = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total = 0.0
        n = 0
        for batch in train_loader:
            x = batch["x"].to(DEVICE)
            y = batch["y"].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total += float(loss.item()) * len(y)
            n += len(y)

        train_loss = total / max(n, 1)
        val_loss = mean_loss(model, val_loader)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"Window {window_id:02d} | Epoch {epoch:02d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

        if val_loss < best_loss - MIN_DELTA:
            best_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f"Early stop at epoch {epoch}, best_epoch={best_epoch}")
                break

    assert best_state is not None, "No model checkpoint was selected"
    model.load_state_dict(best_state)
    return model, train_losses, val_losses, best_epoch


@torch.no_grad()
def predict_model(model: nn.Module, dataset: SequenceDataset) -> pd.DataFrame:
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
    model.eval()
    rows = []
    for batch in loader:
        x = batch["x"].to(DEVICE)
        pred_scaled = model(x).detach().cpu().numpy()
        sample_ids = batch["sample_id"].cpu().numpy()
        pred = pred_scaled * dataset.target_scale
        for sid, yhat in zip(sample_ids, pred):
            meta = dataset.meta[int(sid)]
            rows.append(
                {
                    "secID": meta["secID"],
                    "tradeDate": meta["tradeDate"],
                    "predict_time": meta["predict_time"],
                    "label_time": meta["label_time"],
                    "y_true": meta["y_true"],
                    "y_pred": float(yhat),
                    "feature_input_end_date": meta["feature_input_end_date"],
                }
            )
    return pd.DataFrame(rows)


def mse_metric(predictions: pd.DataFrame) -> float:
    valid = predictions["y_true"].notna() & predictions["y_pred"].notna()
    if not valid.any():
        return float("nan")
    err = predictions.loc[valid, "y_pred"].to_numpy(dtype="float64") - predictions.loc[valid, "y_true"].to_numpy(dtype="float64")
    return float(np.mean(err ** 2))


## 6. Walk-forward 训练函数

按 V2 口径：

```text
504 个交易日训练 / 5 个交易日 embargo / 1 个自然月测试
```

每个窗口内单独拟合 scaler、winsorize 边界和 Transformer 参数。


In [11]:
def run_walk_forward(frame: pd.DataFrame):
    feature_complete = frame[FEATURE_COLUMNS].notna().all(axis=1)
    calendar = pd.DatetimeIndex(frame["tradeDate"].drop_duplicates().sort_values())
    first_usable_date = pd.Timestamp(frame.loc[feature_complete, "tradeDate"].min())
    windows = v1._monthly_windows(calendar, first_usable_date)
    if not windows:
        raise ValueError("Insufficient dates for walk-forward windows")

    print("Total windows:", len(windows))
    print("First window:", windows[0])
    print("Last window:", windows[-1])

    predictions = []
    audits = []

    for window_id, window in enumerate(windows, start=1):
        window_start = time.perf_counter()
        print("\n" + "=" * 100)
        print(f"Window {window_id}/{len(windows)} | test_month={window['test_month']}")
        print(window)

        test_date_mask = frame["tradeDate"].between(window["test_start"], window["test_end"])
        test_mask = test_date_mask & feature_complete
        test = frame.loc[test_mask]
        if test.empty:
            print("Skip empty test window")
            continue

        first_test_predict_time = test["predict_time"].min()
        train_date_mask = frame["tradeDate"].between(window["train_start"], window["train_end"])
        train_mask = (
            train_date_mask
            & feature_complete
            & frame[LABEL_COL].notna()
            & frame[LABEL_TIME_COL].notna()
            & frame[LABEL_TIME_COL].lt(first_test_predict_time)
        )
        train = frame.loc[train_mask]
        if train.empty:
            print("Skip empty train window")
            continue

        assert train["tradeDate"].nunique() == v1.TRAIN_DAYS
        assert train[LABEL_TIME_COL].max() < first_test_predict_time

        y_full = train[LABEL_COL].to_numpy(dtype="float64", copy=True)
        lower, upper = np.quantile(y_full, [0.01, 0.99])
        y_clipped = np.clip(y_full, lower, upper)
        target_scale = float(np.nanstd(y_clipped, ddof=1))
        if not np.isfinite(target_scale) or target_scale <= 0:
            target_scale = 1.0

        scaler = StandardScaler()
        scaler.fit(train[FEATURE_COLUMNS].to_numpy(dtype="float64"))

        train_dates = pd.DatetimeIndex(train["tradeDate"].drop_duplicates().sort_values())
        val_count = max(1, int(math.ceil(len(train_dates) * VALIDATION_DATE_FRACTION)))
        val_start = pd.Timestamp(train_dates[-val_count])
        fit_mask = train_mask & frame["tradeDate"].lt(val_start)
        val_mask = train_mask & frame["tradeDate"].ge(val_start)

        max_train_samples = None if MAX_TRAIN_SAMPLES_PER_WINDOW <= 0 else MAX_TRAIN_SAMPLES_PER_WINDOW

        fit_ds = SequenceDataset(
            frame,
            allowed_mask=fit_mask,
            scaler=scaler,
            target_scale=target_scale,
            label_lower=lower,
            label_upper=upper,
            require_label=True,
            max_samples=max_train_samples,
            random_state=RANDOM_STATE + window_id,
        )
        val_ds = SequenceDataset(
            frame,
            allowed_mask=val_mask,
            scaler=scaler,
            target_scale=target_scale,
            label_lower=lower,
            label_upper=upper,
            require_label=True,
            max_samples=None,
            random_state=RANDOM_STATE + window_id,
        )
        test_ds = SequenceDataset(
            frame,
            allowed_mask=test_mask,
            scaler=scaler,
            target_scale=target_scale,
            label_lower=None,
            label_upper=None,
            require_label=False,
            max_samples=None,
            random_state=RANDOM_STATE + window_id,
        )

        print("train rows full:", int(train_mask.sum()))
        print("fit sequence rows:", len(fit_ds))
        print("validation sequence rows:", len(val_ds))
        print("test sequence rows:", len(test_ds))

        if len(fit_ds) == 0 or len(val_ds) == 0 or len(test_ds) == 0:
            print("Skip because sequence dataset is empty")
            continue

        fit_start = time.perf_counter()
        model, train_losses, val_losses, best_epoch = fit_model(fit_ds, val_ds, window_id)
        fit_seconds = time.perf_counter() - fit_start

        predict_start = time.perf_counter()
        out = predict_model(model, test_ds)
        predict_seconds = time.perf_counter() - predict_start
        out["window_id"] = window_id
        out["train_start"] = window["train_start"]
        out["train_end"] = window["train_end"]
        predictions.append(out)

        total_seconds = time.perf_counter() - window_start
        audits.append(
            {
                **window,
                "window_id": window_id,
                "train_rows_full": int(train_mask.sum()),
                "fit_sequence_rows": len(fit_ds),
                "validation_sequence_rows": len(val_ds),
                "test_sequence_rows": len(test_ds),
                "sequence_length": SEQUENCE_LENGTH,
                "feature_columns": json.dumps(FEATURE_COLUMNS, ensure_ascii=False),
                "label_horizon": LABEL_HORIZON,
                "winsor_1pct": lower,
                "winsor_99pct": upper,
                "target_scale": target_scale,
                "best_epoch": best_epoch,
                "epochs_run": len(train_losses),
                "best_validation_loss": min(val_losses),
                "final_train_loss": train_losses[-1],
                "final_validation_loss": val_losses[-1],
                "train_loss_curve": json.dumps(train_losses),
                "validation_loss_curve": json.dumps(val_losses),
                "fit_seconds": fit_seconds,
                "predict_seconds": predict_seconds,
                "total_window_seconds": total_seconds,
                "device": str(DEVICE),
                "torch_version": torch.__version__,
                "d_model": D_MODEL,
                "n_heads": N_HEADS,
                "n_layers": N_LAYERS,
                "dropout": DROPOUT,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "max_train_samples_per_window": MAX_TRAIN_SAMPLES_PER_WINDOW,
            }
        )

        print(
            f"Completed window {window_id}: best_epoch={best_epoch}, "
            f"fit_seconds={fit_seconds:.1f}, predict_seconds={predict_seconds:.1f}, total={total_seconds:.1f}"
        )

        del model, fit_ds, val_ds, test_ds
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    if not predictions:
        raise RuntimeError("No prediction windows completed")

    result = pd.concat(predictions, ignore_index=True)
    result.sort_values(["tradeDate", "secID"], kind="mergesort", inplace=True)
    result.reset_index(drop=True, inplace=True)

    assert not result.duplicated(["tradeDate", "secID"]).any()
    assert result["feature_input_end_date"].lt(result["predict_time"].dt.normalize()).all()

    return result, pd.DataFrame(audits)


## 7. 正式运行 Transformer walk-forward

这个 Cell 会真的训练模型，耗时取决于电脑性能。第一次可以先保持默认抽样 `MAX_TRAIN_SAMPLES_PER_WINDOW = 120000`。


In [12]:
start_time = time.perf_counter()

predictions, audits = run_walk_forward(featured)

total_time = time.perf_counter() - start_time
print("\nAll windows completed.")
print("Predictions shape:", predictions.shape)
print("Audits shape:", audits.shape)
print("Total seconds:", total_time)

predictions.head()


Total windows: 47
First window: {'test_month': '2022-04', 'train_start': Timestamp('2020-02-28 00:00:00'), 'train_end': Timestamp('2022-03-24 00:00:00'), 'embargo_start': Timestamp('2022-03-25 00:00:00'), 'embargo_end': Timestamp('2022-03-31 00:00:00'), 'test_start': Timestamp('2022-04-01 00:00:00'), 'test_end': Timestamp('2022-04-29 00:00:00')}
Last window: {'test_month': '2026-02', 'train_start': Timestamp('2023-12-26 00:00:00'), 'train_end': Timestamp('2026-01-23 00:00:00'), 'embargo_start': Timestamp('2026-01-26 00:00:00'), 'embargo_end': Timestamp('2026-01-30 00:00:00'), 'test_start': Timestamp('2026-02-02 00:00:00'), 'test_end': Timestamp('2026-02-02 00:00:00')}

Window 1/47 | test_month=2022-04
{'test_month': '2022-04', 'train_start': Timestamp('2020-02-28 00:00:00'), 'train_end': Timestamp('2022-03-24 00:00:00'), 'embargo_start': Timestamp('2022-03-25 00:00:00'), 'embargo_end': Timestamp('2022-03-31 00:00:00'), 'test_start': Timestamp('2022-04-01 00:00:00'), 'test_end': Timesta

train rows full: 2092341
fit sequence rows: 120000
validation sequence rows: 230251
test sequence rows: 101077
Window 08 | Epoch 01 | train_loss=1.021182 | val_loss=0.911539
Window 08 | Epoch 02 | train_loss=1.016641 | val_loss=0.909062
Window 08 | Epoch 03 | train_loss=1.015306 | val_loss=0.904795
Window 08 | Epoch 04 | train_loss=1.014174 | val_loss=0.904439
Window 08 | Epoch 05 | train_loss=1.012285 | val_loss=0.905866
Window 08 | Epoch 06 | train_loss=1.011275 | val_loss=0.910509
Window 08 | Epoch 07 | train_loss=1.009192 | val_loss=0.910865
Early stop at epoch 7, best_epoch=4
Completed window 8: best_epoch=4, fit_seconds=82.5, predict_seconds=2.1, total=131.9

Window 9/47 | test_month=2022-12
{'test_month': '2022-12', 'train_start': Timestamp('2020-10-29 00:00:00'), 'train_end': Timestamp('2022-11-23 00:00:00'), 'embargo_start': Timestamp('2022-11-24 00:00:00'), 'embargo_end': Timestamp('2022-11-30 00:00:00'), 'test_start': Timestamp('2022-12-01 00:00:00'), 'test_end': Timestamp('

train rows full: 2235156
fit sequence rows: 120000
validation sequence rows: 240740
test sequence rows: 100501
Window 16 | Epoch 01 | train_loss=1.013216 | val_loss=0.844073
Window 16 | Epoch 02 | train_loss=1.009803 | val_loss=0.840874
Window 16 | Epoch 03 | train_loss=1.008378 | val_loss=0.843315
Window 16 | Epoch 04 | train_loss=1.006061 | val_loss=0.841846
Window 16 | Epoch 05 | train_loss=1.004867 | val_loss=0.843638
Early stop at epoch 5, best_epoch=2
Completed window 16: best_epoch=2, fit_seconds=60.3, predict_seconds=2.1, total=110.4

Window 17/47 | test_month=2023-08
{'test_month': '2023-08', 'train_start': Timestamp('2021-06-28 00:00:00'), 'train_end': Timestamp('2023-07-24 00:00:00'), 'embargo_start': Timestamp('2023-07-25 00:00:00'), 'embargo_end': Timestamp('2023-07-31 00:00:00'), 'test_start': Timestamp('2023-08-01 00:00:00'), 'test_end': Timestamp('2023-08-31 00:00:00')}
train rows full: 2252056
fit sequence rows: 120000
validation sequence rows: 242036
test sequence row

train rows full: 2348044
fit sequence rows: 120000
validation sequence rows: 247324
test sequence rows: 103013
Window 24 | Epoch 01 | train_loss=0.952356 | val_loss=1.418745
Window 24 | Epoch 02 | train_loss=0.948485 | val_loss=1.417191
Window 24 | Epoch 03 | train_loss=0.946782 | val_loss=1.380131
Window 24 | Epoch 04 | train_loss=0.945905 | val_loss=1.378931
Window 24 | Epoch 05 | train_loss=0.944097 | val_loss=1.375834
Window 24 | Epoch 06 | train_loss=0.941741 | val_loss=1.388655
Window 24 | Epoch 07 | train_loss=0.939517 | val_loss=1.373928
Window 24 | Epoch 08 | train_loss=0.936136 | val_loss=1.357421
Window 24 | Epoch 09 | train_loss=0.933919 | val_loss=1.386428
Window 24 | Epoch 10 | train_loss=0.932918 | val_loss=1.347813
Window 24 | Epoch 11 | train_loss=0.931892 | val_loss=1.377323
Window 24 | Epoch 12 | train_loss=0.929588 | val_loss=1.360990
Completed window 24: best_epoch=10, fit_seconds=148.2, predict_seconds=2.3, total=198.7

Window 25/47 | test_month=2024-04
{'test_mon

Window 31 | Epoch 01 | train_loss=1.005235 | val_loss=0.859691
Window 31 | Epoch 02 | train_loss=0.992437 | val_loss=0.867457
Window 31 | Epoch 03 | train_loss=0.986305 | val_loss=0.879680
Window 31 | Epoch 04 | train_loss=0.977087 | val_loss=0.883455
Early stop at epoch 4, best_epoch=1
Completed window 31: best_epoch=1, fit_seconds=48.4, predict_seconds=2.0, total=98.2

Window 32/47 | test_month=2024-11
{'test_month': '2024-11', 'train_start': Timestamp('2022-09-21 00:00:00'), 'train_end': Timestamp('2024-10-24 00:00:00'), 'embargo_start': Timestamp('2024-10-25 00:00:00'), 'embargo_end': Timestamp('2024-10-31 00:00:00'), 'test_start': Timestamp('2024-11-01 00:00:00'), 'test_end': Timestamp('2024-11-29 00:00:00')}
train rows full: 2422988
fit sequence rows: 120000
validation sequence rows: 247024
test sequence rows: 101365
Window 32 | Epoch 01 | train_loss=0.951749 | val_loss=1.454609
Window 32 | Epoch 02 | train_loss=0.941200 | val_loss=1.457337
Window 32 | Epoch 03 | train_loss=0.934

train rows full: 2449881
fit sequence rows: 120000
validation sequence rows: 248163
test sequence rows: 112509
Window 40 | Epoch 01 | train_loss=1.014157 | val_loss=0.786324
Window 40 | Epoch 02 | train_loss=1.004132 | val_loss=0.787089
Window 40 | Epoch 03 | train_loss=0.996415 | val_loss=0.806320
Window 40 | Epoch 04 | train_loss=0.986237 | val_loss=0.798868
Early stop at epoch 4, best_epoch=1
Completed window 40: best_epoch=1, fit_seconds=48.2, predict_seconds=2.5, total=99.0

Window 41/47 | test_month=2025-08
{'test_month': '2025-08', 'train_start': Timestamp('2023-06-28 00:00:00'), 'train_end': Timestamp('2025-07-24 00:00:00'), 'embargo_start': Timestamp('2025-07-25 00:00:00'), 'embargo_end': Timestamp('2025-07-31 00:00:00'), 'test_start': Timestamp('2025-08-01 00:00:00'), 'test_end': Timestamp('2025-08-29 00:00:00')}
train rows full: 2452997
fit sequence rows: 120000
validation sequence rows: 248750
test sequence rows: 102714
Window 41 | Epoch 01 | train_loss=1.041843 | val_loss=

,secID,tradeDate,predict_time,label_time,y_true,y_pred,feature_input_end_date,window_id,train_start,train_end
0,000001.XSHE,2022-04-01,2022-04-01 15:00:00,2022-04-06 15:00:00,0.040635,-0.000162,2022-03-31,1,2020-02-28,2022-03-24
1,000002.XSHE,2022-04-01,2022-04-01 15:00:00,2022-04-06 15:00:00,0.027106,-0.000961,2022-03-31,1,2020-02-28,2022-03-24
2,000004.XSHE,2022-04-01,2022-04-01 15:00:00,2022-04-06 15:00:00,0.024029,-0.000832,2022-03-31,1,2020-02-28,2022-03-24
3,000006.XSHE,2022-04-01,2022-04-01 15:00:00,2022-04-06 15:00:00,0.031365,-0.001344,2022-03-31,1,2020-02-28,2022-03-24
4,000008.XSHE,2022-04-01,2022-04-01 15:00:00,2022-04-06 15:00:00,0.023529,0.000618,2022-03-31,1,2020-02-28,2022-03-24


## 8. 保存预测结果和窗口审计表

In [13]:
predictions.to_parquet(PREDICTIONS_PATH, index=False)
audits.to_parquet(WINDOWS_PATH, index=False)

print("Saved predictions:", PREDICTIONS_PATH)
print("Saved window audit:", WINDOWS_PATH)

display(audits[[
    "window_id", "test_month", "fit_sequence_rows", "validation_sequence_rows", "test_sequence_rows",
    "best_epoch", "best_validation_loss", "final_train_loss", "final_validation_loss", "total_window_seconds"
]].head())


Saved predictions: /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/artifacts/transformer_1d_predictions.parquet
Saved window audit: /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/artifacts/transformer_1d_windows.parquet


,window_id,test_month,fit_sequence_rows,validation_sequence_rows,test_sequence_rows,best_epoch,best_validation_loss,final_train_loss,final_validation_loss,total_window_seconds
0,1,2022-04,120000,219700,81799,5,1.098470,0.953515,1.116164,146.855833
1,2,2022-05,120000,221288,82397,1,1.192598,0.965248,1.207591,95.079173
2,3,2022-06,120000,220429,92639,1,1.324019,0.958648,1.335900,95.322937
3,4,2022-07,120000,221656,93321,2,1.202695,0.970988,1.209418,107.174823
4,5,2022-08,120000,224524,102976,4,0.898981,0.996470,0.907072,130.447077


## 9. 复用 V2 统一 evaluator 评估结果

In [14]:
def evaluation_input(predictions: pd.DataFrame) -> pd.DataFrame:
    return predictions[["secID", "tradeDate", "y_true", "y_pred", "predict_time", "label_time"]].copy()


eval_result = evaluate_predictions(evaluation_input(predictions), cost_bps=5)
eval_result["name"] = f"transformer_{LABEL_HORIZON}"

mse = mse_metric(predictions)

summary_table = pd.DataFrame([
    {
        "model": f"transformer_{LABEL_HORIZON}",
        "evaluated_rows": eval_result["data_quality"]["evaluated_rows"],
        "valid_days": eval_result["ic_stats"]["valid_days"],
        "rank_ic_mean": eval_result["ic_stats"]["mean"],
        "rank_ic_std": eval_result["ic_stats"]["std"],
        "icir": eval_result["ic_stats"]["icir"],
        "ic_positive_ratio": eval_result["ic_stats"]["positive_ratio"],
        "direction_accuracy": eval_result["direction_stats"]["mean_daily_accuracy"],
        "mse": mse,
        "top_annualized_return": eval_result["annualized_portfolio_stats"]["top"]["annualized_return"],
        "bottom_annualized_return": eval_result["annualized_portfolio_stats"]["bottom"]["annualized_return"],
        "spread_gross_annualized_return": eval_result["annualized_portfolio_stats"]["spread_gross"]["annualized_return"],
        "spread_net_annualized_return": eval_result["annualized_portfolio_stats"]["spread_net"]["annualized_return"],
        "net_annualized_return": eval_result["backtest_metrics"]["annualized_return"],
        "net_annualized_volatility": eval_result["backtest_metrics"]["annualized_volatility"],
        "sharpe": eval_result["backtest_metrics"]["sharpe"],
        "max_drawdown": eval_result["backtest_metrics"]["max_drawdown"],
        "mean_top_turnover": eval_result["backtest_metrics"]["mean_top_turnover"],
        "mean_bottom_turnover": eval_result["backtest_metrics"]["mean_bottom_turnover"],
    }
])

display(summary_table.T)


2026-08-12 23:28:21,116 INFO Evaluation rows: input=4446206 evaluated=4440862 skipped=5344 (y_true_nan=5344, y_pred_nan=0, nonfinite_true=0, nonfinite_pred=0)


,0
model,transformer_1d
evaluated_rows,4440862
valid_days,931
rank_ic_mean,0.038227
rank_ic_std,0.142991
icir,0.26734
ic_positive_ratio,0.604726
direction_accuracy,0.502133
mse,0.001129
top_annualized_return,0.323004


## 10. 生成 V2 evaluator 图和 Transformer loss 曲线

In [15]:
figure_paths = plot_evaluation(eval_result)
print("Evaluator figures:")
for k, v in figure_paths.items():
    print(k, v)


Evaluator figures:
ic /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/artifacts/figures/transformer_1d_rank_ic.png
groups /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/artifacts/figures/transformer_1d_group_returns.png
nav /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/artifacts/figures/transformer_1d_nav.png


In [16]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
for row in audits.itertuples(index=False):
    train_loss = json.loads(row.train_loss_curve)
    plt.plot(range(1, len(train_loss) + 1), train_loss, alpha=0.35)
plt.title("Transformer Training Loss by Window")
plt.xlabel("Epoch")
plt.ylabel("Scaled MSE")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(LOSS_FIGURE_PATH, dpi=150)
plt.show()

print("Loss figure saved:", LOSS_FIGURE_PATH)


Loss figure saved: /Users/runtianzhou/Turing_AI_v5/V1+V2/V2/artifacts/figures/transformer_1d_loss_curves.png


## 11. 写出 Markdown 总结报告

In [17]:
def fmt(value, percent=False):
    if not np.isfinite(value):
        return "NaN"
    return f"{value:.2%}" if percent else f"{value:.6f}"

summary_text = f"""# Transformer Baseline Summary

## Experiment

- Model: Transformer Encoder sequence regressor
- Label horizon: `{LABEL_HORIZON}` (`{LABEL_COL}`)
- Sequence length: {SEQUENCE_LENGTH} trading observations per stock
- Features: {', '.join(FEATURE_COLUMNS)}
- Walk-forward: {v1.TRAIN_DAYS} training days / {v1.EMBARGO_DAYS} embargo days / monthly test
- Scaler and winsorization bounds fitted inside each training window only
- Training rows per window: capped at {MAX_TRAIN_SAMPLES_PER_WINDOW if MAX_TRAIN_SAMPLES_PER_WINDOW > 0 else 'all'} sequences
- Device: {DEVICE}

## Test results

| Metric | Value |
|---|---:|
| Valid evaluated rows | {eval_result['data_quality']['evaluated_rows']:,} |
| Test days | {eval_result['ic_stats']['valid_days']:,} |
| Rank IC mean | {fmt(eval_result['ic_stats']['mean'])} |
| Rank IC std | {fmt(eval_result['ic_stats']['std'])} |
| ICIR | {fmt(eval_result['ic_stats']['icir'])} |
| IC positive ratio | {fmt(eval_result['ic_stats']['positive_ratio'], True)} |
| Direction accuracy | {fmt(eval_result['direction_stats']['mean_daily_accuracy'], True)} |
| MSE | {fmt(mse)} |
| Top annualized return | {fmt(eval_result['annualized_portfolio_stats']['top']['annualized_return'], True)} |
| Bottom annualized return | {fmt(eval_result['annualized_portfolio_stats']['bottom']['annualized_return'], True)} |
| Top-Bottom gross annualized return | {fmt(eval_result['annualized_portfolio_stats']['spread_gross']['annualized_return'], True)} |
| Top-Bottom net annualized return | {fmt(eval_result['annualized_portfolio_stats']['spread_net']['annualized_return'], True)} |
| Net annualized compound return | {fmt(eval_result['backtest_metrics']['annualized_return'], True)} |
| Net annualized volatility | {fmt(eval_result['backtest_metrics']['annualized_volatility'], True)} |
| Sharpe | {fmt(eval_result['backtest_metrics']['sharpe'])} |
| Max drawdown | {fmt(eval_result['backtest_metrics']['max_drawdown'], True)} |
| Mean top turnover | {fmt(eval_result['backtest_metrics']['mean_top_turnover'], True)} |
| Mean bottom turnover | {fmt(eval_result['backtest_metrics']['mean_bottom_turnover'], True)} |

## Output files

- Predictions: `{PREDICTIONS_PATH}`
- Window audit: `{WINDOWS_PATH}`
- Summary: `{SUMMARY_PATH}`
- Loss curves: `{LOSS_FIGURE_PATH}`

## Interpretation note

This is a sequence-model baseline. It should be compared against `ridge_baseline_v2`, `mlp_baseline`, and other V2 outputs under the same evaluator before making any conclusion. A useful Transformer result should improve not only validation loss, but also out-of-sample Rank IC, net spread return, Sharpe, turnover, and drawdown under the shared evaluation contract.
"""

SUMMARY_PATH.write_text(summary_text, encoding="utf-8")
print(summary_text)
print("Saved summary:", SUMMARY_PATH)


# Transformer Baseline Summary

## Experiment

- Model: Transformer Encoder sequence regressor
- Label horizon: `1d` (`label_1d_raw`)
- Sequence length: 20 trading observations per stock
- Features: return_5d, return_10d, return_20d, turnover_20d_mean, log_neg_market_value
- Walk-forward: 504 training days / 5 embargo days / monthly test
- Scaler and winsorization bounds fitted inside each training window only
- Training rows per window: capped at 120000 sequences
- Device: mps

## Test results

| Metric | Value |
|---|---:|
| Valid evaluated rows | 4,440,862 |
| Test days | 931 |
| Rank IC mean | 0.038227 |
| Rank IC std | 0.142991 |
| ICIR | 0.267340 |
| IC positive ratio | 60.47% |
| Direction accuracy | 50.21% |
| MSE | 0.001129 |
| Top annualized return | 32.30% |
| Bottom annualized return | -3.16% |
| Top-Bottom gross annualized return | 35.46% |
| Top-Bottom net annualized return | 29.30% |
| Net annualized compound return | 32.39% |
| Net annualized volatility | 15.68% |
| Sha

## 12. 可选：运行未来 5 日收益版本

上面默认是 V2 的主线任务：预测下一交易日收益。如果要求更贴近 V1 的未来 5 日收益预测，可以重新打开 Notebook，把配置 Cell 里的：

```python
LABEL_HORIZON = "1d"
```

改成：

```python
LABEL_HORIZON = "5d"
```

然后从配置 Cell 开始重新运行。
